In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!pip install -U transformers[sentencepiece] datasets fsspec evaluate huggingface_hub sacrebleu rouge_score py7zr -q

In [ ]:
#Just to disable the weights and biases
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
"""
!pip install --upgrade accelerate
!pip uninstall -v transformers accelerate
!pip install transformers accelerate
"""

'\n!pip install --upgrade accelerate\n!pip uninstall -v transformers accelerate\n!pip install transformers accelerate\n'

In [ ]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
import evaluate
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "google/pegasus-cnn_dailymail"

# Loading Tokeinzer and Model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# Load Dataset [Samsum conversations dataset]
dataset = load_dataset("knkarthick/samsum")

# Remove Empty conversations
def is_valid_example(example):
    return isinstance(example["dialogue"], str) and isinstance(example["summary"], str)

valid_dataset = dataset.filter(is_valid_example)


# Tokenize
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(example_batch['dialogue'],max_length=1024,truncation=True)

    target_encodings = tokenizer(example_batch['summary'],max_length=128,truncation=True)

    return {
        'input_ids': input_encodings['input_ids'],
        'attention_mask': input_encodings['attention_mask'],
        'labels': target_encodings["input_ids"]
    }

tokenized_dataset = valid_dataset.map(convert_examples_to_features, batched=True)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Filter:   0%|          | 0/14732 [00:00<?, ? examples/s]

Filter:   0%|          | 0/818 [00:00<?, ? examples/s]

Filter:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

In [ ]:
from transformers import DataCollatorForSeq2Seq
seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model = model)

In [ ]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir = 'pegasus-samsum', num_train_epochs =1, warmup_steps=500,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    weight_decay=0.01, logging_steps=10,
    eval_strategy='steps', eval_steps=500, save_steps=1e6,
    gradient_accumulation_steps=16
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [ ]:
trainer = Trainer(model=model, args=trainer_args, tokenizer=tokenizer,
                  data_collator=seq2seq_data_collator,
                  train_dataset=tokenized_dataset["test"],
                  eval_dataset = tokenized_dataset["validation"])

/tmp/ipython-input-9-82245412.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=trainer_args, tokenizer=tokenizer,


In [ ]:
trainer.train()

Step,Training Loss,Validation Loss


In [ ]:
# Evaluate

def generate_batch_sized_chunks(list_of_elements, batch_size):
  """split the dataset into smaller batches that we can process simultaneously
  Yield successive batch-sized chunks from the list_of_elements."""

  for i in range(0, len(list_of_elements), batch_size):
    yield list_of_elements[i:i+batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                                batch_size=16, device=device,
                                column_text="article",
                                column_summary="highlights"):
  article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
  target_batches =  list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

  for article_batch, target_batch in tqdm(
      zip(article_batches, target_batches), total=len(article_batches)):

      inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                         padding = "max_length", return_tensors="pt")

      summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                                 attention_mask=inputs["attention_mask"].to(device),
                                 length_penalty=0.8, num_beams=8, max_length=128)

      '''parameter for length penalty ensures that the model does not generate sequences'''

      # Finally, we decode the generated texts, replace the token and add the decoded texts with the references to the metric.
      decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                            clean_up_tokenization_spaces=True)
          for s in summaries]

      decoded_summaries = [d.replace(""," ") for d in decoded_summaries]

      metric.add_batch(predictions=decoded_summaries, references=target_batch)

      #Finally compute and return the ROGUE scores
      score = metric.compute()
  return score

In [ ]:
rogue_names = ['rogue1', 'rogue2', 'rogueL', 'rogueLsum']
rogue_metric = evaluate.load("rogue")

In [ ]:
score = calculate_metric_on_test_ds(
    tokenized_dataset['test'][0:10], rogue_metric, trainer.model, tokenizer, batch_size = 2, column_text = "dialogue", column_summary = "summary"
)

rogue_dict = dict((rn,score[rn].mid.fmeasure) for rn in rogue_names)
pd.DataFrame(rogue_dict, index=[f'pegasus'])

In [ ]:
## Save Model
model.save_pretrained("pegasus-samsum-model")

##save tokenizer
tokenizer.save_pretrained("tokenizer")

In [ ]:
# Load
tokenizer = AutoTokenizer.from_pretrained("/content/tokenizer")

In [ ]:
# Prediction
gen_kwargs = {"length_penalty": 0.8, "num_beams":8, "max_length": 128}
sample_text = tokenized_dataset['test'][0]['dialogue']
reference = tokenized_dataset['test'][0]['summary']

pipe = pipeline("summarization", model='pegasus-samsum-model', tokenizer=tokenizer)

##
print("Dialogue")
print(sample_text)

print("\nReference Summary")
print(reference)

print("\nModel Summary")
print(pipe(sample_text, **gen_kwargs)[0]["summary_text"])

